1. Get original annotations. Strip all 'need' labels. Keep pseudonyms, pronouns, descriptive roles.
2. Get Gemini annotations. Import all 'need' labels.
3. From Gemini annotations: Handle links between entities. and needs. Check if in regex entity list. If not, create new, link.

In [76]:
import pandas as pd
import ast
import re
import uuid
import json

In [77]:
entity_labels = [
    "Pronoun",
    "Descriptive Role",
    "Pseudonym",
]

def import_gemini_predictions(row):
    identified_labels = []
    id_lookup = {}

    # Handle regex predictions (label studio json) - only add entity labels
    regex_predictions = ast.literal_eval(row.regex_predictions)[0]['result']
    for pred in regex_predictions:
        if not pred['value']['labels'][0] in entity_labels:
            continue
        entity_id = str(uuid.uuid4())[:8] # Short unique ID for LS
        pred['id'] = entity_id
        id_lookup[pred['value']['text']] = entity_id
        identified_labels.append(pred)

    # Handle gemini predictions (simplified json) - add all AN labels
    gemini_predictions = ast.literal_eval(row.gemini_predictions)['labels']
    for pred in gemini_predictions:
        if not pred['label'].startswith('cat_'):
            continue
        
        # Find start and end indices of target_text in text
        match = re.search(re.escape(pred['text']), row.note_content, re.IGNORECASE)
        if not match:
            print(f"Warning: Could not find '{pred['text']}' in note_content for note_id {row.note_id}")
            continue
        
        entity_id = str(uuid.uuid4())[:8] # Short unique ID for LS
        pred['id'] = entity_id
        id_lookup[pred['text']] = entity_id

        identified_labels.append({
            "id": entity_id,
            "from_name": "need_labels",
            "to_name": "text",
            "type": "labels",
            "value": {
                'start': match.start(),
                'end': match.end(),
                'text': pred['text'],
                'labels': [pred['label']]
            }
        })
    
    # Handle gemini predictions (simplified json) - add relations
    gemini_links = ast.literal_eval(row.gemini_predictions)['links']
    for link in gemini_links:
        from_id = id_lookup.get(link['from'])
        to_id = id_lookup.get(link['to'])
        if not from_id or not to_id:
            print(f"Warning: Could not find entity IDs for link from '{link['from']}' to '{link['to']}' in note_id {row.note_id}")
            continue

        identified_labels.append({
            "from_id": from_id,
            "to_id": to_id,
            "type": "relation",
            "direction": "right"
        })

    return {
        "data": {
            "id": row.note_id,
            "note_content": row.note_content,
        },
        "predictions": [{
            "model_version": "pre_annotated",
            "score": 1.0,
            "result": identified_labels
        }]
    }

In [78]:
predictions = pd.read_csv('data/input/gemini_pre_annotated.tsv', sep='\t')


In [79]:
predictions['note_id'] = predictions['data'].apply(lambda x: ast.literal_eval(x)['id'])
predictions['note_content'] = predictions['data'].apply(lambda x: ast.literal_eval(x)['note_content'])
predictions.head()

,data,regex_predictions,gemini_predictions,note_id,note_content
0,{'note_content': '[Category: Tenancy Managemen...,"[{'model_version': 'pre_annotated', 'score': 1...","{ ""labels"": [ {""text"": ""Abdul Hopkins Tete D"",...",8b69cab2-f1e2-a627-2e97-784fac3fbd8c,[Category: Tenancy Management] Call Summary; C...
1,{'note_content': '[Category: Tenure management...,"[{'model_version': 'pre_annotated', 'score': 1...","{ ""labels"": [ {""text"": ""Mrs Lauren Green"", ""la...",1fb4ee89-9452-289a-9e80-070a2bd67d76,[Category: Tenure management] DO NOT <URL>Mrs ...
2,{'note_content': '[Category: tenureManagement]...,"[{'model_version': 'pre_annotated', 'score': 1...","{ ""labels"": [ {""text"": ""Miss Eleanor Baker"", ""...",4c925d66-5973-4d1f-a346-2dc1877cb314,[Category: tenureManagement] Succession of ten...
3,{'note_content': '[Category: Rents] [Process: ...,"[{'model_version': 'pre_annotated', 'score': 1...","{ ""labels"": [ { ""text"": ""857664"", ""label"": ""Mi...",6358492d-759e-eccb-8fec-df4caf236b66,[Category: Rents] [Process: 857664] No phone n...
4,{'note_content': '[Category: Tenancy Managemen...,"[{'model_version': 'pre_annotated', 'score': 1...","{ ""labels"": [ {""text"": ""Tenant"", ""label"": ""Des...",2be8f497-882a-3c3b-1c87-85f98036bdbc,[Category: Tenancy Management] Tenant attended...


In [80]:
combined_tasks = predictions.apply(import_gemini_predictions, axis=1)

    Taken into account under the room standards or the space standards' to 'client' in note_id fc541764-8bfe-c1b0-0441-a88da13ce755
prison' to 'sons' in note_id fffaa1fc-8e9e-4f6a-8d68-e1737adbe5c2


In [81]:
with open('data/output/pre-annotations-combined.json', 'w') as f:
    json.dump(combined_tasks.tolist(), f, indent=4)